In [4]:
# VFL SHAP - Prediction Notebook
# All imports at the top
import os
import json
import joblib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Import utility functions
from vfl_utils import simplify_label

# Import model classes
from model import VFLModel, PartyMetaModel

# -----------------------------


In [5]:
# 1. Load Saved Model and Metadata
# -----------------------------
# joblib and Path are imported in cell 0

# Model directory
MODEL_DIR = Path("model")

print("="*80)
print("LOADING SAVED MODEL AND METADATA")
print("="*80)

# Check if model directory exists
if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Model directory '{MODEL_DIR}' not found! Please train the model first using VFL_SHAP_MultiClass.ipynb")

# Load metadata first (contains all configuration)
metadata_path = MODEL_DIR / "model_metadata.json"
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f"✓ Metadata loaded from {metadata_path}")

# Extract configuration from metadata
party1_features = metadata['party1_features']
party2_features = metadata['party2_features']
party3_features = metadata['party3_features']
label_mapping_dict = {int(k): v for k, v in metadata['label_mapping'].items()}  # Convert keys back to int
num_classes = metadata['num_classes']
embed_dim = metadata['embed_dim']
hidden_dim = metadata['hidden_dim']
party_names = metadata['party_names']
party_domains = metadata['party_domains']
party_actions = metadata['party_actions']
party_action_mapping = metadata['party_action_mapping']

print(f"✓ Configuration loaded: {num_classes} classes, embed_dim={embed_dim}, hidden_dim={hidden_dim}")

# Load scalers
scaler1_path = MODEL_DIR / "scaler1.pkl"
scaler2_path = MODEL_DIR / "scaler2.pkl"
scaler3_path = MODEL_DIR / "scaler3.pkl"

scaler1 = joblib.load(scaler1_path)
scaler2 = joblib.load(scaler2_path)
scaler3 = joblib.load(scaler3_path)
print(f"✓ Scalers loaded from {scaler1_path}, {scaler2_path}, {scaler3_path}")

# Model classes are now imported from model.py
# LocalEncoder, ActiveClassifier, VFLModel, and PartyMetaModel are available

# Load VFL model
vfl_model_path = MODEL_DIR / "vfl_model_best.pth"
vfl_checkpoint = torch.load(vfl_model_path, map_location='cpu')
vfl_config = vfl_checkpoint['model_config']

model = VFLModel(
    input_dims=vfl_config['input_dims'],
    embed_dim=vfl_config['embed_dim'],
    num_classes=vfl_config['num_classes'],
    hidden_dim=vfl_config['hidden_dim']
)
model.load_state_dict(vfl_checkpoint['model_state_dict'])
model.eval()
print(f"✓ VFL model loaded from {vfl_model_path}")
print(f"  Best epoch: {vfl_checkpoint['best_epoch']}, Val F1: {vfl_checkpoint['best_val_f1']:.4f}")

# Load meta-model
meta_model_path = MODEL_DIR / "meta_model_best.pth"
meta_checkpoint = torch.load(meta_model_path, map_location='cpu')
meta_config = meta_checkpoint['model_config']

meta_model = PartyMetaModel(
    in_dim=meta_config['in_dim'],
    num_classes=meta_config['num_classes'],
    hidden_dim=meta_config['hidden_dim']
)
meta_model.load_state_dict(meta_checkpoint['model_state_dict'])
meta_model.eval()
print(f"✓ Meta-model loaded from {meta_model_path}")

# Load SHAP background
shap_background_path = MODEL_DIR / "shap_background.npy"
shap_background = np.load(shap_background_path)
print(f"✓ SHAP background loaded from {shap_background_path} (shape: {shap_background.shape})")

# Initialize SHAP explainer
def meta_predict(x_np):
    x_t = torch.tensor(x_np, dtype=torch.float32)
    with torch.no_grad():
        logits = meta_model(x_t)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    return probs

explainer = shap.KernelExplainer(meta_predict, shap_background)
print(f"✓ SHAP explainer initialized")

print("\n" + "="*80)
print("MODEL LOADING SUMMARY")
print("="*80)
print(f"✓ All components loaded successfully!")
print(f"  - VFL Model: {len(party1_features) + len(party2_features) + len(party3_features)} total features")
print(f"  - Meta Model: {meta_config['in_dim']} input dims, {num_classes} classes")
print(f"  - Label mapping: {len(label_mapping_dict)} classes")
print(f"  - Parties: {len(party_names)}")
for i, name in enumerate(party_names):
    print(f"    Party {i+1}: {name} ({len([party1_features, party2_features, party3_features][i])} features)")
print("="*80)
print("Model ready for prediction!")
print("="*80)
# -----------------------------

LOADING SAVED MODEL AND METADATA
✓ Metadata loaded from model\model_metadata.json
✓ Configuration loaded: 9 classes, embed_dim=64, hidden_dim=128
✓ Scalers loaded from model\scaler1.pkl, model\scaler2.pkl, model\scaler3.pkl
✓ VFL model loaded from model\vfl_model_best.pth
  Best epoch: 195, Val F1: 0.9400
✓ Meta-model loaded from model\meta_model_best.pth
✓ SHAP background loaded from model\shap_background.npy (shape: (100, 192))
✓ SHAP explainer initialized

MODEL LOADING SUMMARY
✓ All components loaded successfully!
  - VFL Model: 88 total features
  - Meta Model: 192 input dims, 9 classes
  - Label mapping: 9 classes
  - Parties: 3
    Party 1: Volume_Rate_Sensor_Party1 (23 features)
    Party 2: Timing_Direction_Sensor_Party2 (27 features)
    Party 3: Timing_Direction_Sensor_Party3 (38 features)
Model ready for prediction!


In [6]:
# 2. Load Sample Data and Predict
# -----------------------------
# Configuration
INPUT_DIR = Path("inputs")
RAG_RESULTS_DIR = Path("outputs/predictions")
RAG_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("LOADING SAMPLE DATA AND PREDICTING")
print("="*80)

# Find CSV files in inputs folder
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory '{INPUT_DIR}' not found! Please create it and add your sample CSV file(s).")

csv_files = list(INPUT_DIR.glob("*.csv"))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in {INPUT_DIR}. Please add your sample CSV file(s) to the inputs folder.")

# Use the first CSV file found (or you can specify a pattern)
SAMPLE_CSV_PATH = csv_files[0]
if len(csv_files) > 1:
    print(f"⚠ Multiple CSV files found in {INPUT_DIR}. Using: {SAMPLE_CSV_PATH.name}")
    print(f"  Available files: {[f.name for f in csv_files]}")

# Load sample CSV file
sample_df = pd.read_csv(SAMPLE_CSV_PATH)
print(f"✓ Loaded {len(sample_df)} rows from {SAMPLE_CSV_PATH}")

# Drop unnecessary columns
sample_df = sample_df.drop(columns=["Flow ID", "Src IP", "Dst IP", "Timestamp"], errors="ignore")

# Validate that all required features exist in sample data
print("\nValidating required features...")
all_required_features = party1_features + party2_features + party3_features
missing_features = [feat for feat in all_required_features if feat not in sample_df.columns]

if missing_features:
    print(f"❌ ERROR: {len(missing_features)} required features are missing from sample.csv:")
    for feat in missing_features[:20]:  # Show first 20 missing features
        print(f"  - {feat}")
    if len(missing_features) > 20:
        print(f"  ... and {len(missing_features) - 20} more")
    print(f"\nRequired features: {len(all_required_features)}")
    print(f"Found in sample.csv: {len(all_required_features) - len(missing_features)}")
    print(f"Available columns in sample.csv: {len(sample_df.columns)}")
    print(f"\nSample of available columns (first 10):")
    for col in sample_df.columns[:10]:
        print(f"  - {col}")
    if len(sample_df.columns) > 10:
        print(f"  ... and {len(sample_df.columns) - 10} more")
    print(f"\nPlease ensure sample.csv contains all features used during training.")
    print(f"Common issues:")
    print(f"  1. Column name mismatches (case sensitivity, spaces, etc.)")
    print(f"  2. Missing feature columns")
    print(f"  3. Different dataset format")
    raise KeyError(f"Missing {len(missing_features)} required features in sample.csv. See list above.")

print(f"✓ All {len(all_required_features)} required features found in sample.csv")

# Helper function to partition and preprocess
def partition_and_preprocess_sample(sample_df, party1_features, party2_features, party3_features, 
                                     scaler1, scaler2, scaler3):
    """Partition and preprocess sample data same as training"""
    # Validate features exist (should already be checked, but double-check for safety)
    for party_idx, party_feats in enumerate([party1_features, party2_features, party3_features], 1):
        missing = [f for f in party_feats if f not in sample_df.columns]
        if missing:
            raise KeyError(f"Party {party_idx} missing features: {missing[:5]}")
    
    X1_raw = sample_df[party1_features].values
    X2_raw = sample_df[party2_features].values
    X3_raw = sample_df[party3_features].values
    
    X1 = torch.tensor(scaler1.transform(X1_raw), dtype=torch.float32)
    X2 = torch.tensor(scaler2.transform(X2_raw), dtype=torch.float32)
    X3 = torch.tensor(scaler3.transform(X3_raw), dtype=torch.float32)
    
    return X1, X2, X3

# Preprocess sample data
X1_sample, X2_sample, X3_sample = partition_and_preprocess_sample(
    sample_df, party1_features, party2_features, party3_features, scaler1, scaler2, scaler3
)
x_sample_parts = [X1_sample, X2_sample, X3_sample]
print(f"✓ Preprocessed {len(x_sample_parts[0])} samples")

# Check for labels (optional)
y_sample = None
if "label" in sample_df.columns:
    try:
        sample_df['label_simplified'] = sample_df['label'].apply(simplify_label)
        sample_df['label_numeric'] = sample_df['label_simplified'].map(
            {v: k for k, v in label_mapping_dict.items()}
        )
        y_sample = torch.tensor(sample_df['label_numeric'].values, dtype=torch.long)
        print(f"✓ Labels found: {len(y_sample)} samples with ground truth")
    except Exception as e:
        print(f"⚠ Could not process labels: {e}")

# 3. Predict on All Samples and Save Results
# -----------------------------
print("\n" + "="*80)
print("PREDICTING ON ALL SAMPLES")
print("="*80)

n_samples = len(x_sample_parts[0])
results = []

print(f"Processing {n_samples} samples...")
model.eval()
meta_model.eval()

for idx in range(n_samples):
    if (idx + 1) % 50 == 0 or idx == 0:
        print(f"  Processing sample {idx + 1}/{n_samples}...")
    
    # Get sample
    X1_s = x_sample_parts[0][idx:idx+1]
    X2_s = x_sample_parts[1][idx:idx+1]
    X3_s = x_sample_parts[2][idx:idx+1]
    
    # Get true label if available
    if y_sample is not None:
        true_label_idx = int(y_sample[idx].item())
        true_label = label_mapping_dict.get(true_label_idx, f"Class_{true_label_idx}")
    else:
        true_label_idx = None
        true_label = "UNKNOWN"
    
    # Predict
    with torch.no_grad():
        embeddings = model.get_party_embeddings([X1_s, X2_s, X3_s])
        X_meta = torch.cat(embeddings, dim=1)
        logits = meta_model(X_meta)
        probs = torch.softmax(logits, dim=1)
        predicted_class_idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0, predicted_class_idx].item()
    
    predicted_label = label_mapping_dict.get(predicted_class_idx, f"Class_{predicted_class_idx}")
    
    # Get all probabilities
    all_probs = {label_mapping_dict.get(i, f"Class_{i}"): float(probs[0, i].item()) 
                 for i in range(num_classes)}
    
    # Compute SHAP (party-level)
    X_meta_np = X_meta.detach().cpu().numpy()
    shap_vals_sample = explainer.shap_values(X_meta_np, nsamples=50)
    
    if isinstance(shap_vals_sample, list):
        shap_vals = shap_vals_sample[predicted_class_idx][0]
    else:
        shap_vals = shap_vals_sample[0]
    
    # Aggregate to party level
    embed_dim = 64
    party_shap = []
    for i in range(3):
        start = i * embed_dim
        end = (i + 1) * embed_dim
        party_shap.append(float(np.sum(np.abs(shap_vals[start:end]))))
    
    total_shap = sum(party_shap)
    party_shap_pct = [p / total_shap if total_shap > 0 else 0 for p in party_shap]
    dominant_party_idx = int(np.argmax(party_shap))
    dominant_party = party_names[dominant_party_idx]
    
    # Compute feature-level SHAP for each party
    feature_shap_contributions = {}
    party_feature_lists = [party1_features, party2_features, party3_features]
    party_data = [X1_s, X2_s, X3_s]
    
    # Create wrapper function for party-level feature SHAP
    def create_party_predictor(party_idx, fixed_embeddings):
        """Create a prediction function for a specific party's features"""
        def party_predict(X_party_np):
            """Predict using party features, keeping other parties fixed"""
            X_party_t = torch.tensor(X_party_np, dtype=torch.float32)
            batch_size = X_party_t.shape[0]
            
            model.eval()
            meta_model.eval()
            with torch.no_grad():
                # Get embedding for this party
                party_embedding = model.encoders[party_idx](X_party_t)
                
                # Combine with fixed embeddings from other parties
                all_embeddings = []
                for i, fixed_emb in enumerate(fixed_embeddings):
                    if i == party_idx:
                        all_embeddings.append(party_embedding)
                    else:
                        # Expand fixed embedding to match batch size
                        if fixed_emb.shape[0] == 1:
                            expanded_emb = fixed_emb.repeat(batch_size, 1)
                        else:
                            expanded_emb = fixed_emb[:batch_size] if fixed_emb.shape[0] >= batch_size else fixed_emb
                        all_embeddings.append(expanded_emb)
                
                X_meta_combined = torch.cat(all_embeddings, dim=1)
                logits = meta_model(X_meta_combined)
                probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            return probs
        return party_predict
    
    # Get fixed embeddings for other parties
    with torch.no_grad():
        all_embeddings_fixed = model.get_party_embeddings([X1_s, X2_s, X3_s])
    
    # Compute feature-level SHAP for each party
    for party_idx in range(3):
        party_name = party_names[party_idx]
        party_feat_list = party_feature_lists[party_idx]
        X_party_sample = party_data[party_idx]
        X_party_np = X_party_sample.detach().cpu().numpy()
        
        try:
            # Create background for this party (use sample with small noise)
            bg_size_party = 10
            current_feat = X_party_np[0]
            background_party = np.array([current_feat + np.random.normal(0, 0.1, size=current_feat.shape) 
                                        for _ in range(bg_size_party)])
            
            # Create predictor for this party
            party_predict_fn = create_party_predictor(party_idx, all_embeddings_fixed)
            
            # Compute SHAP values for this party's features
            explainer_party = shap.KernelExplainer(party_predict_fn, background_party)
            shap_values_party = explainer_party.shap_values(X_party_np, nsamples=50, silent=True)
            
            # Handle multi-class SHAP output
            if isinstance(shap_values_party, list):
                shap_vals_party = shap_values_party[predicted_class_idx][0]
            else:
                shap_vals_party = shap_values_party[0] if len(shap_values_party.shape) > 1 else shap_values_party
            
            shap_vals_party = np.array(shap_vals_party).flatten()
            total_abs_shap_party = np.sum(np.abs(shap_vals_party))
            
            # Store feature-level contributions
            feature_contribs = {}
            for feat_idx, feat_name in enumerate(party_feat_list):
                if feat_idx < len(shap_vals_party):
                    shap_val = float(shap_vals_party[feat_idx])
                    abs_shap_val = float(np.abs(shap_val))
                    pct_contrib = (abs_shap_val / total_abs_shap_party * 100.0) if total_abs_shap_party > 0 else 0.0
                    
                    feature_contribs[feat_name] = {
                        "shap_value": shap_val,
                        "abs_shap_value": abs_shap_val,
                        "pct_contribution": pct_contrib
                    }
            
            feature_shap_contributions[party_name] = feature_contribs
            
        except Exception as e:
            # If SHAP computation fails, store empty dict
            feature_shap_contributions[party_name] = {}
    
    # Store result with full structure
    results.append({
        "sample_id": idx,
        "true_label": true_label,
        "true_label_idx": int(true_label_idx) if true_label_idx is not None else None,
        "predicted_label": predicted_label,
        "predicted_label_idx": int(predicted_class_idx),
        "confidence": float(confidence),
        "all_probabilities": all_probs,
        "is_correct": (true_label == predicted_label) if true_label != "UNKNOWN" else None,
        "shap_explanation": {
            "party_contributions": {
                party_names[0]: float(party_shap[0]),
                party_names[1]: float(party_shap[1]),
                party_names[2]: float(party_shap[2])
            },
            "party_contributions_pct": {
                party_names[0]: float(party_shap_pct[0]),
                party_names[1]: float(party_shap_pct[1]),
                party_names[2]: float(party_shap_pct[2])
            },
            "dominant_party": dominant_party,
            "dominant_party_idx": int(dominant_party_idx),
            "dominant_contribution": float(party_shap[dominant_party_idx]),
            "dominant_contribution_pct": float(party_shap_pct[dominant_party_idx]),
            "total_contribution": float(total_shap),
            "feature_contributions": feature_shap_contributions
        },
        "timestamp": datetime.now().isoformat()
    })

print(f"✓ Completed predictions for {len(results)} samples")

# Calculate accuracy if labels available
if y_sample is not None:
    accuracy = sum(1 for r in results if r["is_correct"]) / len(results)
    print(f"✓ Accuracy: {accuracy:.2%}")

# 4. Save Results to RAG_docs/results Folder
# -----------------------------
print("\n" + "="*80)
print("SAVING RESULTS TO RAG_DOCS/RESULTS FOLDER")
print("="*80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Save predictions summary CSV (matches RAG_docs format)
summary_data = []
for r in results:
    summary_data.append({
        "sample_id": r["sample_id"],
        "true_label": r["true_label"],
        "predicted_label": r["predicted_label"],
        "confidence": r["confidence"],
        "is_correct": r["is_correct"],
        "dominant_party": r["shap_explanation"]["dominant_party"],
        "dominant_contribution_pct": r["shap_explanation"]["dominant_contribution_pct"]
    })
summary_df = pd.DataFrame(summary_data)
summary_file = RAG_RESULTS_DIR / f"sample_predictions_summary_{timestamp}.csv"
summary_df.to_csv(summary_file, index=False)
print(f"✓ Predictions summary saved to: {summary_file}")

# 2. Save detailed JSON (matches RAG_docs format)
detailed_file = RAG_RESULTS_DIR / f"sample_predictions_detailed_{timestamp}.json"
with open(detailed_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"✓ Detailed predictions saved to: {detailed_file}")

# 3. Save SHAP explanations CSV (party-level contributions)
shap_data = []
for r in results:
    shap_expl = r["shap_explanation"]
    shap_data.append({
        "sample_id": r["sample_id"],
        "predicted_label": r["predicted_label"],
        "party_1_contrib": shap_expl["party_contributions"][party_names[0]],
        "party_1_contrib_pct": shap_expl["party_contributions_pct"][party_names[0]],
        "party_2_contrib": shap_expl["party_contributions"][party_names[1]],
        "party_2_contrib_pct": shap_expl["party_contributions_pct"][party_names[1]],
        "party_3_contrib": shap_expl["party_contributions"][party_names[2]],
        "party_3_contrib_pct": shap_expl["party_contributions_pct"][party_names[2]],
        "dominant_party": shap_expl["dominant_party"],
        "dominant_contribution": shap_expl["dominant_contribution"],
        "dominant_contribution_pct": shap_expl["dominant_contribution_pct"]
    })
shap_df = pd.DataFrame(shap_data)
shap_file = RAG_RESULTS_DIR / f"sample_shap_explanations_{timestamp}.csv"
shap_df.to_csv(shap_file, index=False)
print(f"✓ SHAP explanations saved to: {shap_file}")

# 4. Save feature-level SHAP contributions CSV
feature_shap_data = []
for r in results:
    sample_id = r["sample_id"]
    predicted_label = r["predicted_label"]
    feature_contribs = r["shap_explanation"]["feature_contributions"]
    
    for party_name, party_features in feature_contribs.items():
        for feat_name, feat_data in party_features.items():
            feature_shap_data.append({
                "sample_id": sample_id,
                "predicted_label": predicted_label,
                "party": party_name,
                "feature_name": feat_name,
                "shap_value": feat_data.get("shap_value", 0.0),
                "abs_shap_value": feat_data.get("abs_shap_value", 0.0),
                "pct_contribution": feat_data.get("pct_contribution", 0.0)
            })
feature_shap_df = pd.DataFrame(feature_shap_data)
feature_shap_file = RAG_RESULTS_DIR / f"sample_feature_shap_contributions_{timestamp}.csv"
feature_shap_df.to_csv(feature_shap_file, index=False)
print(f"✓ Feature SHAP contributions saved to: {feature_shap_file}")

# 5. Save metadata JSON (matches RAG_docs format)
metadata_output = {
    "total_samples": len(results),
    "sample_indices": [r["sample_id"] for r in results],
    "party_names": party_names,
    "label_mapping": {str(k): v for k, v in label_mapping_dict.items()},
    "generation_timestamp": datetime.now().isoformat(),
    "model_info": {
        "num_classes": num_classes,
        "embed_dim": embed_dim
    },
    "feature_partitioning": {
        "party1_features": party1_features,
        "party2_features": party2_features,
        "party3_features": party3_features,
        "party1_feature_count": len(party1_features),
        "party2_feature_count": len(party2_features),
        "party3_feature_count": len(party3_features)
    }
}
metadata_file = RAG_RESULTS_DIR / f"sample_predictions_metadata_{timestamp}.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata_output, f, indent=2, ensure_ascii=False)
print(f"✓ Metadata saved to: {metadata_file}")

print("\n" + "="*80)
print("PREDICTION COMPLETE")
print("="*80)
print(f"All results saved to: {RAG_RESULTS_DIR}")
print(f"  - Summary CSV: {summary_file.name}")
print(f"  - Detailed JSON: {detailed_file.name}")
print(f"  - SHAP Explanations CSV: {shap_file.name}")
print(f"  - Feature SHAP Contributions CSV: {feature_shap_file.name}")
print(f"  - Metadata JSON: {metadata_file.name}")
print("="*80)
# -----------------------------

# -----------------------------

LOADING SAMPLE DATA AND PREDICTING
✓ Loaded 102 rows from inputs\sample.csv

Validating required features...
✓ All 88 required features found in sample.csv
✓ Preprocessed 102 samples
✓ Labels found: 102 samples with ground truth

PREDICTING ON ALL SAMPLES
Processing 102 samples...
  Processing sample 1/102...


  0%|          | 0/1 [00:00<?, ?it/s]

D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 7 iterations, i.e. alpha=1.199e-02, with an active set of 7 regressors, and the smallest cholesky pivot element being 8.429e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 9 iterations, i.e. alpha=5.994e-03, with an active set of 9 regressors, and the smallest cholesky pivot element being 5.960e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 9 iterations, i.e. alpha=5.994e-03, with an active set of 9 regressors, and the smallest cholesky piv

  0%|          | 0/1 [00:00<?, ?it/s]

D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 5 iterations, i.e. alpha=2.844e-02, with an active set of 4 regressors, and the smallest cholesky pivot element being 2.220e-16. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 3 iterations, i.e. alpha=2.370e-05, with an active set of 3 regressors, and the smallest cholesky pivot element being 8.429e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 9 iterations, i.e. alpha=6.372e-03, with an active set of 9 regressors, and the smallest cholesky piv

...

  0%|          | 0/1 [00:00<?, ?it/s]

D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 1 iterations, i.e. alpha=6.792e-02, with an active set of 1 regressors, and the smallest cholesky pivot element being 8.429e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 3 iterations, i.e. alpha=3.394e-02, with an active set of 3 regressors, and the smallest cholesky pivot element being 2.220e-16. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 5 iterations, i.e. alpha=3.287e-02, with an active set of 5 regressors, and the smallest cholesky piv

  0%|          | 0/1 [00:00<?, ?it/s]

D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 1 iterations, i.e. alpha=1.757e-01, with an active set of 1 regressors, and the smallest cholesky pivot element being 2.220e-16. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 9 iterations, i.e. alpha=4.802e-02, with an active set of 8 regressors, and the smallest cholesky pivot element being 5.960e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 1 iterations, i.e. alpha=4.586e-05, with an active set of 1 regressors, and the smallest cholesky piv

✓ Completed predictions for 102 samples
✓ Accuracy: 100.00%

SAVING RESULTS TO RAG_DOCS/RESULTS FOLDER
✓ Predictions summary saved to: outputs\predictions\sample_predictions_summary_20260206_113348.csv
✓ Detailed predictions saved to: outputs\predictions\sample_predictions_detailed_20260206_113348.json
✓ SHAP explanations saved to: outputs\predictions\sample_shap_explanations_20260206_113348.csv
✓ Feature SHAP contributions saved to: outputs\predictions\sample_feature_shap_contributions_20260206_113348.csv
✓ Metadata saved to: outputs\predictions\sample_predictions_metadata_20260206_113348.json

PREDICTION COMPLETE
All results saved to: outputs\predictions
  - Summary CSV: sample_predictions_summary_20260206_113348.csv
  - Detailed JSON: sample_predictions_detailed_20260206_113348.json
  - SHAP Explanations CSV: sample_shap_explanations_20260206_113348.csv
  - Feature SHAP Contributions CSV: sample_feature_shap_contributions_20260206_113348.csv
  - Metadata JSON: sample_predictions_met